In [21]:
# Install rank_bm25 if not already
!pip install rank_bm25

import os
import nltk
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from collections import Counter
import math

# Download NLTK tokenizer
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
folder_path = '/content/drive/MyDrive/Trump Speechs'
query_file_path = '/content/drive/MyDrive/queries1.txt'
# Verify paths exist
print("Checking if paths exist...\n")

if os.path.exists(folder_path):
    txt_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    print(f"✓ Folder found: {folder_path}")
    print(f"  Contains {len(txt_files)} .txt files")
else:
    print(f"✗ Folder NOT found: {folder_path}")
    print("  Please update the folder_path variable above")

print()

if os.path.exists(query_file_path):
    print(f"✓ Query file found: {query_file_path}")
else:
    print(f"✗ Query file NOT found: {query_file_path}")
    print("  Please update the query_file_path variable above")

Checking if paths exist...

✓ Folder found: /content/drive/MyDrive/Trump Speechs
  Contains 56 .txt files

✓ Query file found: /content/drive/MyDrive/queries1.txt


In [11]:
documents = []
doc_names = []

for file in os.listdir(folder_path):
    if file.endswith('.txt'):
        path = os.path.join(folder_path, file)
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read()
            documents.append(text)
            doc_names.append(file)

print(f"Loaded {len(documents)} documents.")


Loaded 56 documents.


In [12]:
with open(query_file_path, 'r', encoding='utf-8') as f:
    queries = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(queries)} queries.")


Loaded 30 queries.


In [20]:
nltk.download('punkt_tab')
tokenized_docs = [word_tokenize(doc.lower()) for doc in documents]
print("Documents tokenized.")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Documents tokenized.


In [15]:
bm25 = BM25Okapi(tokenized_docs)
print("BM25 model initialized.")


BM25 model initialized.


In [16]:
# Flatten all documents for collection probability
collection_tokens = [token for doc in tokenized_docs for token in doc]

# Function to compute LM with Jelinek-Mercer smoothing
def lm_jm_score(query_tokens, doc_tokens, collection_tokens, lamb=0.1):
    doc_len = len(doc_tokens)
    collection_len = len(collection_tokens)
    doc_counts = Counter(doc_tokens)
    collection_counts = Counter(collection_tokens)

    score = 0.0
    for term in query_tokens:
        p_doc = doc_counts[term] / doc_len if doc_len > 0 else 0
        p_coll = collection_counts[term] / collection_len if collection_len > 0 else 0
        smoothed_prob = (1 - lamb) * p_doc + lamb * p_coll
        if smoothed_prob > 0:
            score += math.log(smoothed_prob)
    return score

print("Language Model with Jelinek-Mercer ready.")


Language Model with Jelinek-Mercer ready.


In [17]:
# Flatten all documents for collection probability
collection_tokens = [token for doc in tokenized_docs for token in doc]

# Function to compute LM with Jelinek-Mercer smoothing
def lm_jm_score(query_tokens, doc_tokens, collection_tokens, lamb=0.1):
    doc_len = len(doc_tokens)
    collection_len = len(collection_tokens)
    doc_counts = Counter(doc_tokens)
    collection_counts = Counter(collection_tokens)

    score = 0.0
    for term in query_tokens:
        p_doc = doc_counts[term] / doc_len if doc_len > 0 else 0
        p_coll = collection_counts[term] / collection_len if collection_len > 0 else 0
        smoothed_prob = (1 - lamb) * p_doc + lamb * p_coll
        if smoothed_prob > 0:
            score += math.log(smoothed_prob)
    return score

print("Language Model with Jelinek-Mercer ready.")


Language Model with Jelinek-Mercer ready.


In [18]:
top_k = 3  # Number of top documents to show

for query in queries:
    query_tokens = word_tokenize(query.lower())

    # BM25 ranking
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top_indices = bm25_scores.argsort()[::-1][:top_k]

    # LM Jelinek-Mercer ranking
    lm_scores = [lm_jm_score(query_tokens, doc, collection_tokens, lamb=0.1) for doc in tokenized_docs]
    lm_top_indices = sorted(range(len(lm_scores)), key=lambda i: lm_scores[i], reverse=True)[:top_k]

    print(f"\nQuery: {query}")
    print("BM25 Top Documents:")
    for idx in bm25_top_indices:
        print(f"  {doc_names[idx]} (score: {bm25_scores[idx]:.2f})")

    print("LM Jelinek-Mercer Top Documents:")
    for idx in lm_top_indices:
        print(f"  {doc_names[idx]} (score: {lm_scores[idx]:.2f})")



Query: to
BM25 Top Documents:
  speech_3.txt (score: 1.79)
  speech_30.txt (score: 1.79)
  speech_48.txt (score: 1.79)
LM Jelinek-Mercer Top Documents:
  speech_13.txt (score: -3.16)
  speech_48.txt (score: -3.25)
  speech_41.txt (score: -3.27)

Query: america strong
BM25 Top Documents:
  speech_2.txt (score: 2.97)
  speech_13.txt (score: 2.91)
  speech_46.txt (score: 2.79)
LM Jelinek-Mercer Top Documents:
  speech_13.txt (score: -11.52)
  speech_2.txt (score: -12.00)
  speech_46.txt (score: -12.22)

Query: to bring us
BM25 Top Documents:
  speech_0.txt (score: 4.74)
  speech_3.txt (score: 4.71)
  speech_36.txt (score: 4.66)
LM Jelinek-Mercer Top Documents:
  speech_3.txt (score: -15.64)
  speech_16.txt (score: -15.83)
  speech_36.txt (score: -15.88)

Query: white
BM25 Top Documents:
  speech_49.txt (score: 1.33)
  speech_14.txt (score: 1.21)
  speech_27.txt (score: 1.19)
LM Jelinek-Mercer Top Documents:
  speech_49.txt (score: -6.46)
  speech_14.txt (score: -6.72)
  speech_27.txt (sc